# Image Captioning (BLIP2 + Local MT)

## 0) Setup

In [ ]:
!pip -q install torch torchvision transformers sentencepiece sacremoses pillow pandas tqdm accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 13.9 MB/s eta 0:00:00 0:00:01


## 1) Config (Constraints + Paths + Generation Params)

In [ ]:
import os

CFG = {
    # Data
    "TEST_DIR": "/kaggle/input/competitions/super-ai-engineer-ss-6-thai-language-image-captioning/test/test",
    "SAMPLE_SUBMISSION_PATH": "/kaggle/input/competitions/super-ai-engineer-ss-6-thai-language-image-captioning/sample_submission.csv",
    "OUTPUT_PATH": "submission_blip2_localmt.csv",

    # Caption model (stronger; requires more VRAM)
    # "CAPTION_MODEL": "Salesforce/blip2-flan-t5-xl-coco",
    # "CAPTION_MODEL": "Salesforce/blip2-opt-2.7b",
    "CAPTION_MODEL": "Salesforce/blip-image-captioning-base",

    # Local translation model
    # "MT_MODEL": "facebook/nllb-200-distilled-600M",
    "MT_MODEL": "facebook/nllb-200-distilled-1.3B",
    # "MT_MODEL": "facebook/seamless-m4t-v2-large",
    "SRC_LANG": "eng_Latn",
    "TGT_LANG": "tha_Thai",

    # Prompt and generation constraints (caption)
    # "CAPTION_PROMPT": "Describe this image in one detailed sentence.",
    "CAPTION_MAX_NEW_TOKENS": 40,
    "CAPTION_MIN_NEW_TOKENS": 4,
    "CAPTION_NUM_BEAMS": 2,
    "CAPTION_LENGTH_PENALTY": 2,
    "CAPTION_REPETITION_PENALTY": 1.3,
    "CAPTION_NO_REPEAT_NGRAM_SIZE": 3,

    # Translation constraints
    "MT_MAX_NEW_TOKENS": 40,
    "MT_NUM_BEAMS": 5,
    "MT_LENGTH_PENALTY": 2,
    "MT_NO_REPEAT_NGRAM_SIZE": 3,

    # Thai cleanup constraints
    "MAX_THAI_CHARS": 30,
    "MIN_THAI_CHARS": 2,
    "KEEP_END_PUNCT": False,

    # Runtime
    "SEED": 42,
    "BATCH_PRINT_EVERY": 1,
    "USE_FP16_ON_CUDA": True
}

assert os.path.exists(CFG["TEST_DIR"]), f"Missing test dir: {CFG['TEST_DIR']}"
print(CFG)


{'TEST_DIR': '/kaggle/input/competitions/super-ai-engineer-ss-6-thai-language-image-captioning/test/test', 'SAMPLE_SUBMISSION_PATH': '/kaggle/input/competitions/super-ai-engineer-ss-6-thai-language-image-captioning/sample_submission.csv', 'OUTPUT_PATH': 'submission_blip2_localmt.csv', 'CAPTION_MODEL': 'Salesforce/blip-image-captioning-base', 'MT_MODEL': 'facebook/nllb-200-distilled-600M', 'SRC_LANG': 'eng_Latn', 'TGT_LANG': 'tha_Thai', 'CAPTION_MAX_NEW_TOKENS': 30, 'CAPTION_MIN_NEW_TOKENS': 4, 'CAPTION_NUM_BEAMS': 2, 'CAPTION_LENGTH_PENALTY': 2, 'CAPTION_REPETITION_PENALTY': 1.3, 'CAPTION_NO_REPEAT_NGRAM_SIZE': 3, 'MT_MAX_NEW_TOKENS': 30, 'MT_NUM_BEAMS': 5, 'MT_LENGTH_PENALTY': 2, 'MT_NO_REPEAT_NGRAM_SIZE': 3, 'MAX_THAI_CHARS': 30, 'MIN_THAI_CHARS': 2, 'KEEP_END_PUNCT': False, 'SEED': 42, 'BATCH_PRINT_EVERY': 1, 'USE_FP16_ON_CUDA': True}


## 2) Imports + Utilities

In [ ]:
import re
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm
from transformers import (
    BlipProcessor,
    BlipForConditionalGeneration,
    AutoTokenizer,
    AutoModelForSeq2SeqLM
)


def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def remove_immediate_repetition(text: str) -> str:
    tokens = text.split()
    if not tokens:
        return text
    compact = [tokens[0]]
    for t in tokens[1:]:
        if t != compact[-1]:
            compact.append(t)
    return " ".join(compact)


def thai_cleanup(text: str, max_chars: int = 120, keep_end_punct: bool = False) -> str:
    text = text.strip()
    text = re.sub(r"\s+", " ", text)
    text = remove_immediate_repetition(text)

    replacements = {
        "รูปภาพของ": "ภาพ",
        "ภาพถ่ายของ": "ภาพ",
        "ซึ่งอยู่ใน": "ใน"
    }
    for k, v in replacements.items():
        text = text.replace(k, v)

    if not keep_end_punct:
        text = text.rstrip(" .,!?:;\n\t")

    if len(text) > max_chars:
        text = text[:max_chars].rstrip()

    return text


seed_everything(CFG["SEED"])
device = "cuda" if torch.cuda.is_available() else "cpu"
use_fp16 = (device == "cuda" and CFG["USE_FP16_ON_CUDA"])
print("device:", device, "| use_fp16:", use_fp16)


device: cuda | use_fp16: True


## 3) Load Models

In [ ]:
caption_processor = BlipProcessor.from_pretrained(CFG["CAPTION_MODEL"])

caption_dtype = torch.float16 if use_fp16 else torch.float32
caption_model = BlipForConditionalGeneration.from_pretrained(
    CFG["CAPTION_MODEL"],
    torch_dtype=caption_dtype
).to(device)
caption_model.eval()

mt_tokenizer = AutoTokenizer.from_pretrained(CFG["MT_MODEL"])
mt_model = AutoModelForSeq2SeqLM.from_pretrained(CFG["MT_MODEL"]).to(device)
mt_model.eval()

print("Loaded caption model:", CFG["CAPTION_MODEL"])
print("Loaded translation model:", CFG["MT_MODEL"])


preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Loaded caption model: Salesforce/blip-image-captioning-base
Loaded translation model: facebook/nllb-200-distilled-600M


## 4) Inference Functions

In [ ]:
@torch.inference_mode()
def generate_english_caption(image: Image.Image) -> str:
    inputs = caption_processor(
        images=image,
        # text=CFG["CAPTION_PROMPT"],
        return_tensors="pt"
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    output_ids = caption_model.generate(
        **inputs,
        max_new_tokens=CFG["CAPTION_MAX_NEW_TOKENS"],
        min_new_tokens=CFG["CAPTION_MIN_NEW_TOKENS"],
        num_beams=CFG["CAPTION_NUM_BEAMS"],
        length_penalty=CFG["CAPTION_LENGTH_PENALTY"],
        repetition_penalty=CFG["CAPTION_REPETITION_PENALTY"],
        no_repeat_ngram_size=CFG["CAPTION_NO_REPEAT_NGRAM_SIZE"]
    )

    en = caption_processor.batch_decode(output_ids, skip_special_tokens=True)[0].strip()
    # en = en.replace(CFG["CAPTION_PROMPT"], "").strip(" :\n\t")
    return en

@torch.inference_mode()
def translate_en_to_th(text_en: str) -> str:
    mt_tokenizer.src_lang = CFG["SRC_LANG"]
    batch = mt_tokenizer(
        [text_en],
        return_tensors="pt",
        truncation=True,
        max_length=256
    )
    batch = {k: v.to(device) for k, v in batch.items()}

    output_ids = mt_model.generate(
        **batch,
        forced_bos_token_id=mt_tokenizer.convert_tokens_to_ids(CFG["TGT_LANG"]),
        max_new_tokens=CFG["MT_MAX_NEW_TOKENS"],
        num_beams=CFG["MT_NUM_BEAMS"],
        length_penalty=CFG["MT_LENGTH_PENALTY"],
        no_repeat_ngram_size=CFG["MT_NO_REPEAT_NGRAM_SIZE"]
    )

    return mt_tokenizer.batch_decode(output_ids, skip_special_tokens=True)[0].strip()


def make_thai_caption(image: Image.Image) -> dict:
    en = generate_english_caption(image)
    # print(f"en: {en}")
    th_raw = translate_en_to_th(en)
    # print(f"th_raw: {th_raw}")
    th = thai_cleanup(
        th_raw,
        max_chars=CFG["MAX_THAI_CHARS"],
        keep_end_punct=CFG["KEEP_END_PUNCT"]
    )
    # print(f"th: {th}")
    if len(th) < CFG["MIN_THAI_CHARS"]:
        th = f"ภาพที่มี {th}" if th else "ภาพถ่ายที่มีวัตถุและฉาก"

    return {"en": en, "th": th}


## 5) Run Captioning

In [ ]:
test_dir = Path(CFG["TEST_DIR"])
image_files = sorted([p for p in test_dir.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"}])

rows = []

for i, image_path in enumerate(tqdm(image_files, total=len(image_files))):
    # if i >= 5:
    #     break
    try:
        with Image.open(image_path) as img:
            img = img.convert("RGB")
            out = make_thai_caption(img)

        rows.append({
            "image_filename": image_path.name,
            "caption_en": out["en"],
            "caption_th": out["th"]
        })

        if (i + 1) % CFG["BATCH_PRINT_EVERY"] == 0:
            sample = rows[-1]
            print(f"[{i+1}] {sample['image_filename']} | EN: {sample['caption_en']} | TH: {sample['caption_th']}")

    except RuntimeError as e:
        err = str(e).lower()
        if "out of memory" in err:
            rows.append({
                "image_filename": image_path.name,
                "caption_en": "",
                "caption_th": "ภาพถ่ายที่มีวัตถุและฉาก"
            })
            print(f"OOM on {image_path.name}. Try reducing CAPTION_MAX_NEW_TOKENS/NUM_BEAMS.")
            if device == "cuda":
                torch.cuda.empty_cache()
        else:
            rows.append({
                "image_filename": image_path.name,
                "caption_en": "",
                "caption_th": "ภาพถ่ายที่มีวัตถุและฉาก"
            })
            print(f"RuntimeError on {image_path.name}: {e}")
    except Exception as e:
        rows.append({
            "image_filename": image_path.name,
            "caption_en": "",
            "caption_th": "ภาพถ่ายที่มีวัตถุและฉาก"
        })
        print(f"Error on {image_path.name}: {e}")

pred_df = pd.DataFrame(rows)
pred_df.head()


  0%|          | 0/2000 [00:00<?, ?it/s]

[1] 00000.jpg | EN: a brown and white horse standing on a dirt road | TH: ม้าสีน้ําตาลและขาวยืนอยู่บนถนน
[2] 00001.jpg | EN: a plate of food that includes meat, vegetables, and sauce | TH: จานอาหารที่รวมถึงเนื้อ ผัก หมั
[3] 00002.jpg | EN: a close up of a flower on a plant | TH: รูปแบบใกล้เคียงของดอกไม้บนพืช
[4] 00003.jpg | EN: a close up of a cow ' s nose | TH: รูปแบบใกล้เคียงของจมูกวัว
[5] 00004.jpg | EN: a small dog sitting on top of a table next to a bowl of candy | TH: สุนัขเล็ก ๆ น้อย ๆ ที่นั่งบนโต
[6] 00005.jpg | EN: a close up of a plant with green leaves | TH: รูปแบบใกล้เคียงของพืชที่มีใบเข
[7] 00006.jpg | EN: a small frog sitting on top of a moss covered rock | TH: กวางเล็ก ๆ น้อย ๆ ที่นั่งอยู่บ
[8] 00007.jpg | EN: a picture of a man in a suit and tie standing in front of a door | TH: ภาพชายคนหนึ่งในชุดและกางเกงยืน
[9] 00008.jpg | EN: a small fire in the middle of a dark cave | TH: การเผาไหม้เล็ก ๆ น้อย ๆ ท่ามกล
[10] 00009.jpg | EN: a field of straw straws in the middle of a 

,image_filename,caption_en,caption_th
0,00000.jpg,a brown and white horse standing on a dirt road,ม้าสีน้ําตาลและขาวยืนอยู่บนถนน
1,00001.jpg,"a plate of food that includes meat, vegetables...",จานอาหารที่รวมถึงเนื้อ ผัก หมั
2,00002.jpg,a close up of a flower on a plant,รูปแบบใกล้เคียงของดอกไม้บนพืช
3,00003.jpg,a close up of a cow ' s nose,รูปแบบใกล้เคียงของจมูกวัว
4,00004.jpg,a small dog sitting on top of a table next to ...,สุนัขเล็ก ๆ น้อย ๆ ที่นั่งบนโต


## 6) Quick Quality Check

In [ ]:
display(pred_df.sample(min(10, len(pred_df)), random_state=CFG["SEED"]))


,image_filename,caption_en,caption_th
1860,01860.jpg,a black beetle on the ground in front of a house,มีแมลงดําที่นอนอยู่บนพื้นหน้าบ
353,00353.jpg,a man walking down a dirt road with a sign,ผู้ชายคนหนึ่งเดินตามถนนที่ดินพ
1333,01333.jpg,a bird is perched on a branch in the forest,นกอยู่บนสาขาของป่าในป่า
905,00905.jpg,a white flower with green leaves in the backgr...,ดอกสีขาวที่มีใบเขียวในเบื้องหล
1289,01289.jpg,a flower in the middle of a field,ดอกไม้ที่กลางสนาม
1273,01273.jpg,a person in a wet suit and goggles snoring on ...,คนที่ใส่ชุดชื้นและแว่นตาที่หงื
938,00938.jpg,a field of banana trees with blue sky in the b...,สนามต้นทองแดงที่มีท้องฟ้าฟ้าสี
1731,01731.jpg,a body of water with trees in the background,แผ่นน้ําที่มีต้นไม้อยู่เบื้องห
65,00065.jpg,a trail through the woods in the fog,เส้นทางผ่านป่าในหมอก
1323,01323.jpg,a view of a river in the middle of a forest,วิวแม่น้ํากลางป่า


## 7) Build Submission

In [ ]:
submission = pd.read_csv(CFG["SAMPLE_SUBMISSION_PATH"], dtype=str)
pred_map = dict(zip(pred_df["image_filename"], pred_df["caption_th"]))

for idx, row in submission.iterrows():
    image_id = str(row["image_id"])
    fname = f"{image_id}.jpg"

    if (pd.isna(row["caption"]) or str(row["caption"]).strip() == "") and fname in pred_map:
        submission.at[idx, "caption"] = pred_map[fname]

submission.to_csv(CFG["OUTPUT_PATH"], index=False, encoding="utf-8")
print("Saved:", CFG["OUTPUT_PATH"])
submission.head()


Saved: submission_blip2_localmt.csv


,image_id,caption
0,01354,ภาพถ่ายระยะใกล้ของวัตถุทรงกลมสีขาวที่มีคราบสีด...
1,01413,นกตัวหนึ่งที่กําลังเกาะอยู่บนกิ่งไม้อันหนึ่งที...
2,01802,บ้านที่อยู่ติดกับบริเวณริมชายหาดทะเลและมีต้นไม...
3,01243,ปลาลิงที่นั่งอยู่บนพื้น ติดกํา
4,00693,มีสิงห์สองตัวที่ขยับไปบนพื้น
